# Exercises XP : Évaluer des LLM pour la synthèse de texte (Corrigé)

## Ce que tu vas apprendre
- Évaluer concrètement des LLM sur une tâche de résumé (summarization) : exactitude (accuracy) vs ROUGE.
- Comprendre les forces et les faiblesses de chaque métrique, et comparer plusieurs tailles de modèles.
- Utiliser les bibliothèques Hugging Face `transformers` et `evaluate`.
- Charger, échantillonner et prétraiter des données textuelles, puis déboguer les sorties d'un LLM.

**Ce que tu vas créer** : des scripts d'évaluation, des tableaux comparatifs, une métrique d'exactitude personnalisée, et de courtes analyses écrites.


In [1]:
# ============================================================
# Partie I. Installation (à exécuter une seule fois par session)
# ============================================================
# - rouge_score : calcule la métrique ROUGE
# - evaluate : bibliothèque Hugging Face qui encapsule des métriques comme ROUGE
# - datasets : pour charger le jeu de données CNN/DailyMail
# - transformers : pour charger et utiliser les modèles (T5, GPT-2, ...)
# - accelerate : optimise l'exécution des modèles (utile avec un GPU)
# - nltk : utilisé ici pour découper un texte en phrases (sent_tokenize)
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
# 'punkt' et 'punkt_tab' sont des modèles de découpage de phrases utilisés par nltk.sent_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.1 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

### Partie II. Chargement et exploration du jeu de données

Jeu de données utilisé : [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (on renomme la colonne `article` en `prompt_text`, et `highlights` en `prompt_title`).
- Si tu as des fichiers CSV locaux `train.csv` / `test.csv` avec les colonnes `prompt_text` / `prompt_title`, renseigne les chemins ci-dessous.
- Sinon, on télécharge automatiquement un petit échantillon du jeu de données Hugging Face pour rester léger.
- On affiche quelques lignes pour vérifier que tout est correct.
- Si le téléchargement Hugging Face échoue (pas d'accès Internet, etc.), un tout petit jeu de données de secours (fallback) est utilisé automatiquement.


In [2]:
import pandas as pd
from datasets import load_dataset

# Laisse ces chemins vides pour utiliser l'échantillon Hugging Face (ou le fallback si erreur)
train_path = ''  # ex : '/content/train.csv'
test_path = ''   # ex : '/content/test.csv'

# Petit jeu de données de secours utilisé uniquement si le téléchargement échoue
fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    """Charge un split (train/test) depuis un CSV local, ou depuis Hugging Face si aucun chemin n'est fourni,
    puis en garde un échantillon aléatoire de n lignes (pour que le notebook reste rapide)."""
    if path:
        df = pd.read_csv(path)
    else:
        try:
            # On demande directement les n premières lignes du split voulu (ex: 'train[:100]')
            hf_split = f"{split_name}[:{max(n, 3)}]"
            # Config '3.0.0' = version non anonymisée du dataset, avec les colonnes 'article' et 'highlights'
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(
                columns={'article': 'prompt_text', 'highlights': 'prompt_title'}
            )
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    # random_state=42 fixe le hasard : on obtient toujours le même échantillon si on relance la cellule
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

# On prend 100 exemples d'entraînement et 50 exemples de test, comme demandé dans la consigne
train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

# --- Inspection des données pour un débutant ---
print("Colonnes du DataFrame d'entraînement :", list(train_df.columns))
print("Taille de train_df :", train_df.shape)
print("Taille de test_df :", test_df.shape)

print("\n--- Premier exemple d'entraînement ---")
print("Article (prompt_text), premiers 500 caractères :\n", train_df.loc[0, 'prompt_text'][:500])
print("\nRésumé de référence (prompt_title) :\n", train_df.loc[0, 'prompt_title'])

print("\n--- Aperçu du DataFrame d'entraînement échantillonné ---")
display(train_df.head())

print("\n--- Aperçu du DataFrame de test échantillonné ---")
display(test_df.head())

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Colonnes du DataFrame d'entraînement : ['prompt_text', 'prompt_title']
Taille de train_df : (100, 2)
Taille de test_df : (50, 2)

--- Premier exemple d'entraînement ---
Article (prompt_text), premiers 500 caractères :
 SHANGHAI, China -- Championship leader Lewis Hamilton spun out of the Chinese Grand Prix to send the world title race to a cliffhanger finale in Brazil on October 21. A disconsolate Lewis Hamilton leaves his car after spinning into the gravel trap in Shanghai. Rookie Hamilton can still clinch the crown with third place in the closing race, but saw his 12-point lead cut to four by McLaren team-mate Fernando Alonso in Shanghai. The Spaniard finished second behind Ferrari's Kimi Raikkonen, who also

Résumé de référence (prompt_title) :
 Lewis Hamilton fails to clinch world title after spinning out of the Chinese GP .
Briton's lead cut to four points by McLaren team-mate Fernando Alonso .
Alonso finished second in Shanghai behind Ferrari's Kimi Raikkonen .

--- Aperçu du Dat

,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...



--- Aperçu du DataFrame de test échantillonné ---


,prompt_text,prompt_title
0,(CNN)A year ago Bloomberg published a story wi...,"LZ: Indiana law pushing back LGBT rights, and ..."
1,"(CNN)Mark Ronson's ""Uptown Funk!,"" featuring B...",The song rules the chart for 13th week .\nIt p...
2,"(CNN)Police in the Indian city of Malegaon, in...",Authorities in the Indian city of Malegaon hav...
3,"(CNN)""Jake the dog and Finn the human. The fun...",Thai Airways subsidiary Thai Smile features Ca...
4,"Marseille, France (CNN)The French prosecutor l...","Marseille prosecutor says ""so far no videos we..."


### Partie III. Résumé (summarization) avec T5
Tâches :
- Écrire `batch_generator` pour découper une liste en petits lots (batches).
- Écrire `summarize_with_t5` en utilisant `t5-small` (ou une autre taille) avec le GPU si disponible.
- Préfixer chaque texte d'entrée par `"summarize: "` (c'est ainsi que T5 sait qu'on lui demande un résumé), puis décoder avec `skip_special_tokens=True`.
- Vider le cache CUDA entre les lots (`torch.cuda.empty_cache()`) et appeler `gc.collect()`.


In [3]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    """Découpe la liste 'items' en petits paquets (batches) de taille 'batch_size'.
    Exemple : batch_generator([1,2,3,4,5], 2) renvoie [1,2], puis [3,4], puis [5].
    On traite les données par petits paquets plutôt que d'un coup pour ne pas saturer la mémoire."""
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    """Génère un résumé pour chaque texte de 'texts' en utilisant un modèle T5."""
    # On utilise le GPU si disponible (beaucoup plus rapide), sinon le CPU
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
    model.eval()  # on désactive le mode entraînement, on ne fait que de l'inférence

    summaries: List[str] = []

    for batch_texts in batch_generator(texts, batch_size):
        # T5 est un modèle "text-to-text" : il faut lui dire quelle tâche effectuer avec un préfixe
        inputs = ["summarize: " + t for t in batch_texts]

        # On tokenize le lot d'un coup (padding = les textes courts sont complétés pour avoir la même longueur)
        encoded = tokenizer(
            inputs, return_tensors="pt", padding=True, truncation=True, max_length=512
        ).to(device)

        with torch.no_grad():  # pas besoin de calculer les gradients, on ne fait que générer du texte
            output_ids = model.generate(**encoded, max_new_tokens=max_new_tokens)

        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        summaries.extend(decoded)

        # On libère la mémoire après chaque lot pour éviter les erreurs "out of memory"
        del encoded, output_ids
        if device == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

    if device == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

    return summaries

# RUN_T5 permet de désactiver facilement la génération (opération lente) pendant le débogage
RUN_T5 = True
if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=4)
    display(pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    }).head())
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

,prompt_text,reference_summary,t5_small_summary
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest..."
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...


### Partie IV. Évaluation de l'exactitude (accuracy) — probablement proche de zéro
On calcule une exactitude "naïve" qui vérifie si le résumé généré est EXACTEMENT identique, mot pour mot, au résumé de référence.
On va voir pourquoi cette métrique est beaucoup trop stricte pour du texte généré librement (elle vaut presque toujours zéro).


In [4]:
from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    """Renvoie la proportion de résumés générés qui sont EXACTEMENT identiques (chaîne de caractères) au résumé de référence."""
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")

# Pourquoi cette exactitude est-elle presque toujours nulle ?
# - Un LLM génère du texte librement : il peut reformuler, changer l'ordre des mots, utiliser des synonymes...
# - Il suffit d'UNE SEULE lettre de différence (majuscule, ponctuation, espace) pour que la comparaison échoue.
# - Deux résumés peuvent avoir exactement le même sens mais ne jamais correspondre mot pour mot.
# C'est pour cela qu'on a besoin d'une métrique plus souple comme ROUGE (Partie V).

Exact-match accuracy: 0.0000


### Partie V. Implémentation de la métrique ROUGE
On utilise `evaluate.load("rouge")` et le découpeur de phrases de NLTK.
On prétraite le texte en séparant les phrases par des retours à la ligne, ce qui améliore le calcul de ROUGE-L.


In [5]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

# On charge la métrique ROUGE une seule fois (le chargement peut prendre quelques secondes)
rouge = evaluate.load('rouge')

def normalize_text(text):
    """Découpe le texte en phrases puis les rejoint avec un retour à la ligne entre chaque phrase.
    C'est important pour ROUGE-L, qui repère les plus longues sous-séquences communes : les retours à la ligne
    aident la métrique à bien distinguer les phrases."""
    sents = sent_tokenize(text.strip())
    return "\n".join(sents)  # on utilise '\n' (retour à la ligne) et non '' comme le suggérait le squelette

def compute_rouge_score(preds: List[str], refs: List[str]):
    """Calcule les scores ROUGE (rouge1, rouge2, rougeL, rougeLsum) entre une liste de résumés générés
    et une liste de résumés de référence."""
    preds_norm = [normalize_text(p) for p in preds]
    refs_norm = [normalize_text(r) for r in refs]
    # use_stemmer=True : réduit les mots à leur racine (ex: 'running' -> 'run') avant de comparer,
    # ce qui rend la comparaison moins sensible aux petites variations grammaticales.
    return rouge.compute(predictions=preds_norm, references=refs_norm, use_stemmer=True)

# Test de bon sens ("sanity check") avec des chaînes identiques et une prédiction vide
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check:")
print(compute_rouge_score(test_preds, test_refs))

ROUGE sanity check:
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.6666666666666666), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}


### Partie VI. Comprendre les scores ROUGE
Expériences à mener (les observations sont commentées directement dans le code ci-dessous) :
- Correspondance exacte vs. prédiction vide.
- Effet du stemming : ex. "running" vs. "run".
- Chevauchement de n-grammes : comment ROUGE-1 vs. ROUGE-2 évoluent avec un chevauchement partiel.
- Symétrie : on échange prédictions et références et on compare.


In [6]:
# --- Expérience 1 : correspondance exacte ---
# Quand le résumé généré est identique au résumé de référence, tous les scores ROUGE doivent valoir 1.0 (le maximum).
identical = ["the quick brown fox jumps over the lazy dog"]
print("1) Correspondance exacte :", compute_rouge_score(identical, identical))

# --- Expérience 2 : prédiction vide ---
# Si le modèle ne génère rien, il n'y a aucun mot en commun avec la référence : tous les scores doivent valoir 0.0.
empty_pred = [""]
ref_for_empty = ["the quick brown fox jumps over the lazy dog"]
print("2) Prédiction vide :", compute_rouge_score(empty_pred, ref_for_empty))

1) Correspondance exacte : {'rouge1': np.float64(1.0), 'rouge2': np.float64(1.0), 'rougeL': np.float64(1.0), 'rougeLsum': np.float64(1.0)}
2) Prédiction vide : {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}


In [7]:
# --- Expérience 3 : effet du stemming ---
# Le stemming réduit les mots à leur racine (ex: 'running' -> 'run'), donc deux phrases qui utilisent des formes
# différentes du même mot devraient obtenir un score plus élevé AVEC stemming que SANS.
pred_stem = ["the dogs are running quickly"]
ref_stem = ["the dog was running quickly"]

preds_norm = [normalize_text(p) for p in pred_stem]
refs_norm = [normalize_text(r) for r in ref_stem]

score_with_stemmer = rouge.compute(predictions=preds_norm, references=refs_norm, use_stemmer=True)
score_without_stemmer = rouge.compute(predictions=preds_norm, references=refs_norm, use_stemmer=False)

print("3) Avec stemming   :", score_with_stemmer)
print("   Sans stemming   :", score_without_stemmer)
# Observation attendue : le score AVEC stemming est généralement plus élevé, car 'dogs'/'dog' et les formes de
# 'running' sont ramenées à une racine commune, augmentant le nombre de mots considérés comme "identiques".

3) Avec stemming   : {'rouge1': np.float64(0.8000000000000002), 'rouge2': np.float64(0.5), 'rougeL': np.float64(0.8000000000000002), 'rougeLsum': np.float64(0.8000000000000002)}
   Sans stemming   : {'rouge1': np.float64(0.6), 'rouge2': np.float64(0.25), 'rougeL': np.float64(0.6), 'rougeLsum': np.float64(0.6)}


In [8]:
# --- Expérience 4 : chevauchement de n-grammes (ROUGE-1 vs ROUGE-2) ---
# ROUGE-1 compte les mots individuels en commun (unigrammes).
# ROUGE-2 compte les paires de mots consécutifs en commun (bigrammes), donc il est plus strict :
# il faut que les mots soient dans le MÊME ORDRE pour compter.
reference = "the quick brown fox jumps over the lazy dog"

candidates = [
    "the quick brown fox jumps over the lazy dog",      # chevauchement total
    "the quick brown fox jumps over a sleepy dog",       # chevauchement partiel, ordre conservé
    "dog lazy the over jumps fox brown quick the",       # mêmes mots, ordre complètement mélangé
    "a cat sleeps peacefully on the sofa",                # aucun mot en commun
]

for cand in candidates:
    scores = compute_rouge_score([cand], [reference])
    print(f"Candidat : {cand!r}")
    print(f"  ROUGE-1 = {scores['rouge1']:.3f} | ROUGE-2 = {scores['rouge2']:.3f}\n")

# Observation attendue :
# - Phrase identique : ROUGE-1 et ROUGE-2 valent tous les deux 1.0.
# - Chevauchement partiel (mots changés) : les deux scores baissent, mais ROUGE-2 baisse davantage car les
#   bigrammes autour des mots changés sont "cassés".
# - Mots identiques mais ordre mélangé : ROUGE-1 reste élevé (les mots individuels sont tous présents),
#   mais ROUGE-2 chute fortement (presque aucune paire de mots consécutifs ne correspond).
# - Aucun mot en commun : ROUGE-1 et ROUGE-2 valent 0.0.

Candidat : 'the quick brown fox jumps over the lazy dog'
  ROUGE-1 = 1.000 | ROUGE-2 = 1.000

Candidat : 'the quick brown fox jumps over a sleepy dog'
  ROUGE-1 = 0.778 | ROUGE-2 = 0.625

Candidat : 'dog lazy the over jumps fox brown quick the'
  ROUGE-1 = 1.000 | ROUGE-2 = 0.000

Candidat : 'a cat sleeps peacefully on the sofa'
  ROUGE-1 = 0.125 | ROUGE-2 = 0.000



In [9]:
# --- Expérience 5 : symétrie ---
# ROUGE combine précision et rappel dans un F-score (moyenne harmonique). Si on échange prédiction et référence,
# la précision et le rappel échangent simplement leurs valeurs — et le F-score, lui, reste identique.
short_text = "the quick brown fox"
long_text = "the quick brown fox jumps over the lazy dog in the sunny afternoon"

scores_a = compute_rouge_score([short_text], [long_text])   # short_text = prédiction, long_text = référence
scores_b = compute_rouge_score([long_text], [short_text])   # on échange : long_text = prédiction, short_text = référence

print("Prédiction courte / Référence longue :", scores_a)
print("Prédiction longue / Référence courte :", scores_b)

# Observation attendue : scores_a et scores_b sont EXACTEMENT IDENTIQUES. Avec le F-score par défaut (celui que
# renvoie evaluate.load('rouge')), ROUGE est donc symétrique : échanger prédiction et référence ne change rien.
# Ce ne serait PAS le cas si on isolait uniquement le rappel ou uniquement la précision : ces deux composantes,
# elles, ne sont pas symétriques.

Prédiction courte / Référence longue : {'rouge1': np.float64(0.47058823529411764), 'rouge2': np.float64(0.4), 'rougeL': np.float64(0.47058823529411764), 'rougeLsum': np.float64(0.47058823529411764)}
Prédiction longue / Référence courte : {'rouge1': np.float64(0.47058823529411764), 'rouge2': np.float64(0.4), 'rougeL': np.float64(0.47058823529411764), 'rougeLsum': np.float64(0.47058823529411764)}


### Partie VII. Comparer un petit et un grand modèle
Objectifs :
- Générer des résumés avec `t5-small`, `t5-base`, et `gpt2` (avec un prompt de type "TL;DR").
- Calculer ROUGE pour chacun et stocker les scores ligne par ligne.
- Implémenter `compute_rouge_per_row` pour ajouter les colonnes ROUGE à un DataFrame.
- Implémenter `summarize_with_gpt2` avec un préfixe "TL;DR:" et une gestion prudente de la longueur.

Pour que le notebook reste rapide, on utilise ici un petit sous-échantillon de démonstration (`demo_df`, 10 exemples). Tu peux remplacer `demo_df` par `train_df` en entier si tu as plus de temps ou un GPU.


In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    """Génère un résumé de type 'TL;DR' avec GPT-2 (modèle de génération de texte, pas conçu spécifiquement
    pour le résumé, contrairement à T5)."""
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # GPT-2 n'a pas de token de padding par défaut : on utilise le token de fin de séquence (eos) à la place
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # GPT-2 est un modèle 'décodeur seul' : il doit être complété (padding) À GAUCHE, pas à droite,
    # sinon la génération démarre juste après des tokens de remplissage et perd en qualité.
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    model.eval()

    summaries: List[str] = []

    for batch_texts in batch_generator(texts, batch_size):
        # On tronque l'article (GPT-2 a une limite de contexte courte) puis on ajoute le prompt "TL;DR:"
        # qui invite GPT-2 (entraîné sur du texte web) à produire un résumé court.
        prompts = [t[:800] + "\nTL;DR:" for t in batch_texts]

        encoded = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True, max_length=900
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **encoded,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.pad_token_id
            )

        for ids in output_ids:
            full_text = tokenizer.decode(ids, skip_special_tokens=True)
            # GPT-2 continue simplement le texte : on ne garde que ce qui vient APRÈS "TL;DR:"
            if "TL;DR:" in full_text:
                summary = full_text.split("TL;DR:")[-1].strip()
            else:
                summary = full_text.strip()
            summaries.append(summary)

        del encoded, output_ids
        if device == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

    if device == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

    return summaries

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    """Calcule les scores ROUGE ligne par ligne (un score par exemple, pas seulement une moyenne globale)
    et les ajoute comme nouvelles colonnes au DataFrame."""
    rouge1_list, rouge2_list, rougeL_list, rougeLsum_list = [], [], [], []
    for pred, ref in zip(df[pred_col], df[ref_col]):
        scores = compute_rouge_score([pred], [ref])
        rouge1_list.append(scores['rouge1'])
        rouge2_list.append(scores['rouge2'])
        rougeL_list.append(scores['rougeL'])
        rougeLsum_list.append(scores['rougeLsum'])

    result = df.copy()
    result[f'{pred_col}_rouge1'] = rouge1_list
    result[f'{pred_col}_rouge2'] = rouge2_list
    result[f'{pred_col}_rougeL'] = rougeL_list
    result[f'{pred_col}_rougeLsum'] = rougeLsum_list
    return result

RUN_COMPARE = True
if RUN_COMPARE and 'train_summaries_t5' in locals():
    # Petit sous-échantillon pour garder le notebook rapide (voir la note ci-dessus)
    demo_df = train_df.head(10).reset_index(drop=True)

    demo_summaries_t5_small = summarize_with_t5(demo_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2)
    demo_summaries_t5_base = summarize_with_t5(demo_df['prompt_text'].tolist(), model_name='t5-base', batch_size=2)
    demo_summaries_gpt2 = summarize_with_gpt2(demo_df['prompt_text'].tolist(), model_name='gpt2', batch_size=2)

    demo_df['t5_small_summary'] = demo_summaries_t5_small
    demo_df['t5_base_summary'] = demo_summaries_t5_base
    demo_df['gpt2_summary'] = demo_summaries_gpt2

    demo_df = compute_rouge_per_row(demo_df, 't5_small_summary')
    demo_df = compute_rouge_per_row(demo_df, 't5_base_summary')
    demo_df = compute_rouge_per_row(demo_df, 'gpt2_summary')

    display(demo_df)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

,prompt_text,prompt_title,t5_small_summary,t5_base_summary,gpt2_summary,t5_small_summary_rouge1,t5_small_summary_rouge2,t5_small_summary_rougeL,t5_small_summary_rougeLsum,t5_base_summary_rouge1,t5_base_summary_rouge2,t5_base_summary_rougeL,t5_base_summary_rougeLsum,gpt2_summary_rouge1,gpt2_summary_rouge2,gpt2_summary_rougeL,gpt2_summary_rougeLsum
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,championship leader Lewis Hamilton spun out of...,Hamilton's car was caught in the middle of the...,0.222222,0.131148,0.222222,0.222222,0.548387,0.300000,0.516129,0.516129,0.208955,0.030769,0.119403,0.208955
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...,Xinhua: some children who swallowed the beads ...,The Aqua Dots toys contain a chemical that can...,0.147059,0.000000,0.117647,0.117647,0.208955,0.123077,0.179104,0.208955,0.162162,0.027778,0.108108,0.108108
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...,Qualcomm was founded in 1985 by seven communic...,Qualcomm is a global leader in the mobile indu...,0.424242,0.156250,0.333333,0.393939,0.242424,0.031250,0.181818,0.242424,0.323529,0.060606,0.235294,0.264706
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest...",new: police arrest acting president of ex-Prim...,Musharraf's order to take a television station...,0.237288,0.035088,0.203390,0.169492,0.246154,0.000000,0.215385,0.215385,0.500000,0.285714,0.444444,0.472222
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...,third seed Julia vakulenko will face former wo...,Julia Vakulenko will face comeback queen Linds...,0.413793,0.142857,0.275862,0.413793,0.500000,0.241379,0.300000,0.500000,0.542373,0.210526,0.305085,0.440678
5,"SAN DIEGO, California (CNN) -- More than 100 h...",San Diego mayor declares state of emergency; W...,new: mayor says he received offers of aid from...,landslide pulls earth from beneath three-lane ...,The landslide was caused by a landslide that h...,0.242424,0.031250,0.151515,0.181818,0.171429,0.000000,0.085714,0.171429,0.191781,0.056338,0.164384,0.191781
6,(CNN) -- A former government contract employee...,NEW: Indictment: Man tried to pass nuclear fil...,sources say the classified materials were take...,former contractor indicted on charges of steal...,The U.S. government is at risk of being expose...,0.083333,0.000000,0.055556,0.083333,0.187500,0.064516,0.187500,0.187500,0.164384,0.000000,0.136986,0.164384
7,"HELSINKI, Finland (CNN) -- An 18-year-old auth...","NEW: Teen gunman is dead, Finnish police say ....",police say 18-year-old pekka Eric Auvinen shot...,police: 18-year-old shooter identified as pekk...,A high school student shot and killed in Finla...,0.253968,0.065574,0.190476,0.190476,0.070175,0.000000,0.070175,0.070175,0.238806,0.000000,0.149254,0.208955
8,WASHINGTON (CNN) -- As he awaits a crucial pro...,President Bush to address the Veterans of Fore...,president will argue against pulling out from ...,president bush to talk about why u.s. should w...,Bush's war on Iraq is a mistake.\nJUST WATCHED...,0.426230,0.237288,0.360656,0.426230,0.366667,0.206897,0.300000,0.333333,0.222222,0.032787,0.158730,0.222222
9,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...,he says he has no plans to fritter cash away o...,young actor says he has no plans to fritter hi...,Harry Potter star Daniel Radcliffe has no plan...,0.354839,0.266667,0.322581,0.354839,0.444444,0.360656,0.380952,0.412698,0.412698,0.360656,0.412698,0.412698


### Partie VIII. Comparer tous les modèles
À implémenter :
- `compare_models` : agrège les scores ROUGE moyens de plusieurs modèles dans un seul DataFrame.
- `compare_models_summaries` : affiche les résumés générés par tous les modèles côte à côte.
Affiche les deux tableaux et discute quel modèle gagne, et pourquoi.


In [11]:
import pandas as pd

def compare_models(rouge_dict):
    """Prend un dictionnaire {nom_du_modele: scores_rouge_moyens} et renvoie un DataFrame
    avec une ligne par modèle et une colonne par métrique ROUGE."""
    df = pd.DataFrame(rouge_dict).T  # .T = transpose : les modèles deviennent des lignes
    df.index.name = 'model'
    return df.reset_index()

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    """Renvoie un sous-DataFrame avec le texte source, le résumé de référence, et les résumés générés
    par chaque modèle, pour les comparer facilement côte à côte."""
    cols = ['prompt_text', 'prompt_title'] + pred_cols
    cols = [c for c in cols if c in df.columns]  # on ignore les colonnes qui n'existent pas
    return df[cols]

if RUN_COMPARE and 'demo_df' in locals():
    # Scores ROUGE moyens (agrégés) de chaque modèle sur le sous-échantillon de démonstration
    rouge_dict = {
        't5-small': compute_rouge_score(demo_df['t5_small_summary'].tolist(), demo_df['prompt_title'].tolist()),
        't5-base': compute_rouge_score(demo_df['t5_base_summary'].tolist(), demo_df['prompt_title'].tolist()),
        'gpt2': compute_rouge_score(demo_df['gpt2_summary'].tolist(), demo_df['prompt_title'].tolist()),
    }

    print("--- Scores ROUGE moyens par modèle ---")
    display(compare_models(rouge_dict))

    print("\n--- Résumés générés côte à côte ---")
    display(compare_models_summaries(
        demo_df, ['t5_small_summary', 't5_base_summary', 'gpt2_summary']
    ))

    # Piste de réflexion : t5-base est généralement plus performant que t5-small (plus de paramètres,
    # entraîné pour le résumé), tandis que gpt2 (non spécialisé pour le résumé) obtient souvent des scores
    # ROUGE plus faibles, même si son texte reste grammaticalement correct.

--- Scores ROUGE moyens par modèle ---


,model,rouge1,rouge2,rougeL,rougeLsum
0,t5-small,0.279520,0.107346,0.223499,0.255228
1,t5-base,0.296704,0.132668,0.238413,0.286313
2,gpt2,0.293961,0.106014,0.223833,0.269503



--- Résumés générés côte à côte ---


,prompt_text,prompt_title,t5_small_summary,t5_base_summary,gpt2_summary
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,championship leader Lewis Hamilton spun out of...,Hamilton's car was caught in the middle of the...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...,Xinhua: some children who swallowed the beads ...,The Aqua Dots toys contain a chemical that can...
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...,Qualcomm was founded in 1985 by seven communic...,Qualcomm is a global leader in the mobile indu...
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest...",new: police arrest acting president of ex-Prim...,Musharraf's order to take a television station...
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...,third seed Julia vakulenko will face former wo...,Julia Vakulenko will face comeback queen Linds...
5,"SAN DIEGO, California (CNN) -- More than 100 h...",San Diego mayor declares state of emergency; W...,new: mayor says he received offers of aid from...,landslide pulls earth from beneath three-lane ...,The landslide was caused by a landslide that h...
6,(CNN) -- A former government contract employee...,NEW: Indictment: Man tried to pass nuclear fil...,sources say the classified materials were take...,former contractor indicted on charges of steal...,The U.S. government is at risk of being expose...
7,"HELSINKI, Finland (CNN) -- An 18-year-old auth...","NEW: Teen gunman is dead, Finnish police say ....",police say 18-year-old pekka Eric Auvinen shot...,police: 18-year-old shooter identified as pekk...,A high school student shot and killed in Finla...
8,WASHINGTON (CNN) -- As he awaits a crucial pro...,President Bush to address the Veterans of Fore...,president will argue against pulling out from ...,president bush to talk about why u.s. should w...,Bush's war on Iraq is a mistake.\nJUST WATCHED...
9,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...,he says he has no plans to fritter cash away o...,young actor says he has no plans to fritter hi...,Harry Potter star Daniel Radcliffe has no plan...
